# Theorem 22 — Diffusion necessity under proper scores

**Formal source:** [`../22_diffusion_necessity_under_proper_scores.md`](../22_diffusion_necessity_under_proper_scores.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
rng = np.random.default_rng(22)
samples = rng.normal(0.4, 0.8, 80000)
true_density = normal_pdf(samples, 0.4, 0.8)
richer_density = 0.3 * true_density + 0.7 * true_density
np.testing.assert_allclose(true_density, richer_density, rtol=1e-14, atol=1e-14)
component = rng.integers(0, 2, 80000)
mixture_samples = np.where(component == 0, rng.normal(-2, 0.4, 80000), rng.normal(2, 0.4, 80000))
gaussian_nll = -np.mean(np.log(normal_pdf(mixture_samples, mixture_samples.mean(), mixture_samples.std()) + 1e-300))
mixture_nll = -np.mean(np.log(0.5 * normal_pdf(mixture_samples, -2, 0.4) + 0.5 * normal_pdf(mixture_samples, 2, 0.4) + 1e-300))
assert mixture_nll + 0.5 < gaussian_nll
print({"contained_family_gap": 0.0, "misspecified_gaussian_nll": float(gaussian_nll), "mixture_nll": float(mixture_nll)})

In [ ]:
print('THEORY_DEMO_PASS::22_diffusion_necessity_under_proper_scores')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')